# Lower Bounds

Run this at:
* Nodes 1-10
* Every 2 nodes for nodes 10-50
* Every 10 nodes after that

In [ ]:
# get solution to LP relaxation (I'm assuming this is an np.array or something?) - call this S
# create solution graph (same as you do now) - call this graph G (**see note 1 below)

# remove all edges from G where visited is integer
# for all arcs in G:
    # set arc.visited = visited % 1 (**see note 2 below)
# for all nodes in G:
    # set node.demand = sum(in edges visited) - sum(out edges visited)

# if node.supply == 0 for all nodes in G:
    # Change S by setting x_ij = int(x_ij) for all x_ij in S
# elif number of nodes with demand != 0 is < 10:
    # set flow = nx.min_cost_flow(G,weight="visited") (**see note 3 below)
    # for all arcs (u,v) in G:
        # if u in flow and v in flow[u]:
            # Change S by setting x_ij = flow[u][v]
        # else:
            # Change S by setting x_ij = 0
# else:
    # do nothing, this probably isn't feasible

# if we got a solution:
    # check that it is feasible
    # check that it is better than current lower bound
    # give the feasible solution to gurobi (**see note 4 below)

: 

**Note 1**: I'm assuming that you're using the "visited" attribute to indicate how many times we run an edge in the solution, just like you are right now

**Note 2**: What this line does is get only the fractional part of the visited count. I read the paper again, and this is in fact what we want to do lol

**Note 3**: So, in the documentation for this function, they say it's not guaranteed to work when our edge weights are floats. It _should_ work, because there are some pretty specific circumstances where it doesn't, but if it doesn't work, then just multiply visited by 1000 and convert to an int for every edge.

**Note 4**: I asked chatgpt if this was even possible to do (because it turns out there's not a ton of ways you can modify gurobi's branch-and-bound algorithm), and it helpfully provided the following starter code:

In [ ]:
def my_callback(model, where):
    if where == GRB.Callback.MIPNODE:
        node_count = model.cbGet(GRB.Callback.MIPNODE_NODCNT)

        if (
            node_count <= 10
            or (node_count <= 50 and node_count % 2 == 0)
            or node_count % 10 == 0
        ):
            # Get LP relaxation solution
            relaxation_solution = model.cbGetNodeRel(model.getVars())

            # Your function to adjust values, returns a dictionary
            new_solution = get_lower_bound(relaxation_solution)

            obj_expr = model.getObjective()  # Get the objective function expression
            obj_val = sum(
                new_solution[var] * coeff for var, coeff in obj_expr.getVarCoeff()
            )  # Calculcate objective value for new solution

            # If your feasible solution is better than gurobi's, replace the solution
            if obj_val > model.getObjective():
                model.cbSetSolution(model.getVars(), new_solution)

# Connectivity Cuts

Run this at every node

In [ ]:
# get solution to LP relaxation (I'm assuming this is an np.array or something?) - call this S
# create solution graph (same as you do now) - call this graph G

# for epsilon in [0, 0.25, 0.5]:
    # create a copy of G - call this graph H
    # remove all edges from H where visited < 1-epsilon
    
    # constraints_to_add = []

    # for comp in nx.strongly_connected_components(H):
        # if start node not in comp:
            # in_edges = list of edges (u,v) in G, where u not in comp and v in comp
            # comp_edges = list of edges (u,v) in G, where both u and v are in comp
            # for comp_edge in comp_edges:
                # if sum(in_edge["visited"]) < comp_edge["visited"]:
                    # append constraint sum x_ij >= y_kl to constaints_to_add, where the x_ij are the edges in in_edges and y_kl is the comp_edge
    # len(constraints_to_add) > 0:
        # break
# else: (ie, if we didn't add any constraints for any value of epsilon)
    # if we're at the root branch-and bound node:
        # constraints_to_add = []
        # for each edge (u,v) in G with visited > 0:
            # create a copy of G - call this graph H
            # add a new node w to H
            # add the edge (u,w) to H with visited = G[u][v]["visited"]
            # add the edge (w,v) to H with visited = float("inf")
            # remove edge (u,v) from H
            
            # cut_val, partition = nx.minimum_cut(H, start node, w, capacity="visited")
            # if cut_val < G[u][v]["visited"]:
                # i_set, j_set = partition
                # cutset = set()
                # for i, i_successors in ((i, G[i]) for i in i_set): (this for-loop and cutset update is pulled directly from the networkx docs)
                    # cutset.update((i,j) for j in i_successors if j in j_set)
                # add node v to j_set
                # for comp_edge in nx.induced_subgraph(G, j_set).edges:
                    # append constraint sum x_ij >= y_kl to constraints_to_add, where the x_ij are the edges in cutset and y_kl is comp_edge
                    
# if len(constraint_to_add) > 0:
    # add the constraints as cuts to gurobi